# QQQ Data Pull 

 The raw daily panel via using data from Refinitiv, Yahoo, and FRED.

**Sources:** 
 - ETF prices/volume + Treasury yields from Refinitiv 
 - VIX/VIX3M from Yahoo 
 -  Real yield / Fed Funds / credit spread from FRED (these are macro series, not exchange instruments). 





In [12]:
# ── Imports ──
import os, ssl, time, logging, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
from fredapi import Fred

ssl._create_default_https_context = ssl._create_unverified_context
warnings.filterwarnings('ignore')
logging.getLogger('pyeikon').setLevel(logging.CRITICAL)   # silence Eikon per-block "no data" spam
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50); pd.set_option('display.precision', 4)

In [14]:
#
# DATA — keys, pullers, Refinitiv-primary ladder (Refinitiv -> Yahoo -> FRED), save raw panel CSV
#
HERE = Path.cwd(); DATA = HERE / 'data'; DATA.mkdir(exist_ok=True)
_AD = HERE.parent / 'Anomaly Detection'      # where the shared .env lives
START = '1999-01-01'

def load_keys(required=('FRED_KEY', 'EIKON_KEY')):
    merged = {}
    for p in [_AD / '.env', Path.home() / '.config/anomaly_detection/keys.env']:
        if p.exists():
            for line in p.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith('#') or '=' not in line: continue
                k, v = line.split('=', 1)
                merged.setdefault(k.strip(), v.strip().strip('"').strip("'"))
    keys = {k: os.environ.get(k, merged.get(k)) for k in set(merged) | set(required)}
    missing = [k for k in required if not keys.get(k)]
    if missing: raise ValueError(f'Missing {missing}; looked in {_AD / ".env"}')
    return keys

KEYS = load_keys()
fred = Fred(api_key=KEYS['FRED_KEY'])

_EIKON_READY = None
def _eikon_ok():
    global _EIKON_READY
    if _EIKON_READY is not None: return _EIKON_READY
    if not KEYS.get('EIKON_KEY'): print('No EIKON_KEY'); _EIKON_READY = False; return False
    try:
        import eikon as ek
        ek.set_app_key(KEYS['EIKON_KEY'])
        ek.get_timeseries('QQQ.O', fields=['CLOSE'], start_date='2024-01-02', end_date='2024-01-10', interval='daily')
        _EIKON_READY = True; print('Eikon connected')
    except Exception as e:
        print(f'Eikon unavailable ({str(e)[:55]}) — falling back to Yahoo/FRED'); _EIKON_READY = False
    return _EIKON_READY

def pull_eikon(ric, start=START, sleep=0.5, field='CLOSE', block=5):
    # 5-year blocks (~1260 rows < get_timeseries' ~3000 cap). Refinitiv returns None when throttled —
    # retry None/empty/transient with backoff; only a "No data" EXCEPTION means genuine pre-inception.
    if not _eikon_ok(): return None
    import eikon as ek
    y0, y1 = int(start[:4]), 2026
    chunks = []
    for ys in range(y0, y1 + 1, block):
        ye = min(ys + block - 1, y1)
        for attempt in range(5):
            try:
                df = ek.get_timeseries(rics=ric, fields=[field],
                        start_date=f'{ys}-01-01', end_date=f'{ye}-12-31', interval='daily')
            except Exception as e:
                if 'no data' in str(e).lower(): break          # genuine pre-inception — stop retrying
                time.sleep(sleep * 4); continue                # transient error — retry
            if df is None or getattr(df, 'empty', True):
                time.sleep(sleep * 4); continue                # throttled — retry (this was the bug)
            if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(1)
            s = df[field].squeeze()
            if getattr(s.index, 'tz', None) is not None: s.index = s.index.tz_localize(None)
            s.index = pd.to_datetime(s.index)
            s = pd.to_numeric(s, errors='coerce').dropna()
            if len(s): chunks.append(s)
            break
        time.sleep(sleep)
    if not chunks: return None
    full = pd.concat(chunks).sort_index()
    return full[~full.index.duplicated(keep='first')]

def pull_yahoo(ticker, start=START, col='Close'):
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    if df is None or df.empty: return pd.Series(dtype=float)
    s = df[col]
    if isinstance(s, pd.DataFrame): s = s.iloc[:, 0]
    s.index = pd.to_datetime(s.index)
    if getattr(s.index, 'tz', None) is not None: s.index = s.index.tz_localize(None)
    return pd.to_numeric(s, errors='coerce').dropna().sort_index()

def pull_fred(sid, start=START):
    try:
        s = fred.get_series(sid)
        s.index = pd.to_datetime(s.index)
        s = pd.to_numeric(s, errors='coerce').dropna().sort_index()
        return s[s.index >= pd.Timestamp(start)]
    except Exception as e:
        print(f'  FRED {sid}: FAIL ({str(e)[:60]})'); return pd.Series(dtype=float)

# Refinitiv RICs primary; Yahoo fallback for prices; FRED for pure-macro series
PRICES = {
    'QQQ': dict(ric='QQQ.O', yft='QQQ'),
    'SPY': dict(ric='SPY.P', yft='SPY'),
    'TLT': dict(ric='TLT.O', yft='TLT'),
    'GLD': dict(ric='GLD.P', yft='GLD'),
    'USO': dict(ric='USO.P', yft='USO'),
    'UUP': dict(ric='UUP.P', yft='UUP'),
}
VOL = {
    'VIX':   dict(yft='^VIX',   fred_id='VIXCLS'),             # Refinitiv denies .VIX on this account
    'VIX3M': dict(yft='^VIX3M', fred_id='VXVCLS'),
}
RATES = {
    'US2Y':          dict(ric='US2YT=RR',  fred_id='DGS2'),    # Refinitiv benchmark yield, FRED fallback
    'US10Y':         dict(ric='US10YT=RR', fred_id='DGS10'),
    'real_10y':      dict(fred_id='DFII10'),                   # 10y TIPS real yield (FRED macro)
    'FedFunds':      dict(fred_id='DFF'),
    'credit_spread': dict(fred_id='BAA10Y'),                   # Moody's Baa - 10y (ICE HY OAS API-capped to ~3y)
}

def pull_series(ric=None, yft=None, fred_id=None, start=START):
    if ric:
        s = pull_eikon(ric, start)
        if s is not None and len(s) > 100: return s, f'Refinitiv {ric}'
    if yft:
        s = pull_yahoo(yft, start)
        if len(s) > 100: return s, f'Yahoo {yft}'
    if fred_id:
        for fid in ([fred_id] if isinstance(fred_id, str) else fred_id):
            s = pull_fred(fid, start)
            if len(s) > 100: return s, f'FRED {fid}'
    return pd.Series(dtype=float), 'NONE'

raw, prov = {}, {}
for name, cfg in {**PRICES, **VOL, **RATES}.items():
    raw[name], prov[name] = pull_series(**cfg)

# QQQ trading VOLUME — Refinitiv VOLUME field, Yahoo fallback
qv = pull_eikon('QQQ.O', field='VOLUME'); qv_src = 'Refinitiv QQQ.O[VOLUME]'
if qv is None or len(qv) < 100:
    qv = pull_yahoo('QQQ', col='Volume'); qv_src = 'Yahoo QQQ[Volume]'
raw['QQQ_volume'], prov['QQQ_volume'] = qv, qv_src

base = raw['QQQ'].index
panel = pd.DataFrame(index=base)
for name in raw:
    panel[name] = raw[name].reindex(base).ffill()            # ffill = last known value, no look-ahead
panel['slope_10y2y'] = panel['US10Y'] - panel['US2Y']
prov['slope_10y2y'] = 'derived US10Y-US2Y'
panel.index.name = 'Date'
panel.to_csv(DATA / 'qqq_raw_panel.csv')
pd.DataFrame({'provenance': prov}).to_csv(DATA / 'qqq_provenance.csv')

cov = pd.DataFrame({
    'provenance':  [prov.get(c, '?')                     for c in panel.columns],
    'first_valid': [panel[c].first_valid_index().date()  for c in panel.columns],
    'n_obs':       [int(panel[c].notna().sum())          for c in panel.columns],
}, index=panel.columns)
print('Raw panel saved ->', DATA / 'qqq_raw_panel.csv')
print(panel.shape, '| end:', panel.index[-1].date())
cov

2026-06-15 20:19:16,062 P[94031] [MainThread 8284674240] Error: no proxy address identified.
Check if Eikon Desktop or Eikon API Proxy is running.
2026-06-15 20:19:31,088 P[94031] [MainThread 8284674240] Error on handshake url http://127.0.0.1:9090/api/handshake : ReadTimeout('')
2026-06-15 20:19:31,092 P[94031] [MainThread 8284674240] Error on handshake url http://127.0.0.1:9090/api/handshake : ReadTimeout('')
2026-06-15 20:19:31,093 P[94031] [MainThread 8284674240] Port number was not identified, cannot send any request
2026-06-15 20:19:31,148 P[94031] [MainThread 8284674240] Eikon Proxy not running or cannot be reached. Please read the documentation on troubleshooting


Eikon unavailable (Error code 401 | Eikon Proxy not running or cannot be r) — falling back to Yahoo/FRED
Raw panel saved -> /Users/yellleutenegger/Desktop/CAS_WYLeuteneggerFinal/data/qqq_raw_panel.csv
(6859, 15) | end: 2026-06-15


,provenance,first_valid,n_obs
QQQ,Yahoo QQQ,1999-03-10,6859
SPY,Yahoo SPY,1999-03-10,6859
TLT,Yahoo TLT,2002-07-30,6008
GLD,Yahoo GLD,2004-11-18,5426
USO,Yahoo USO,2006-04-10,5077
UUP,Yahoo UUP,2007-03-01,4854
VIX,Yahoo ^VIX,1999-03-10,6859
VIX3M,Yahoo ^VIX3M,2006-07-17,5010
US2Y,FRED DGS2,1999-03-10,6859
US10Y,FRED DGS10,1999-03-10,6859


note: UUP as limiting factor as it only starts in 2007.